# R13 · Comparación de modelos

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Comparación de candidatos para el score de pago tardío (B), descrita en
[`comparacion_modelos.md`](comparacion_modelos.md). Diseño aprobado en D-26.

| Sección | Criterio de R13 |
|---|---|
| §1 Pliegues | Misma partición; CV agrupada por trimestre (unidad de R12) |
| §2 Candidatos por defecto | ≥ 3 candidatos con media y desviación; línea base; tiempo de ajuste |
| §3 Búsqueda de hiperparámetros | Frente al modelo por defecto |
| §4 Modelo final preliminar | ¿La mejora supera la desviación de la CV? |
| §5 Importancia por permutación | En datos no vistos, frente a la de impureza |
| §6 Grupos | Un grupo perjudicado con su tasa de error |

**La prueba no se usa.** Todo se hace con el entrenamiento completo (2022-01 a 2025-06) y sus
pliegues. **Confidencialidad:** solo cifras agregadas.

## 0. Configuración

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import datos
from cartera import diseno as dis
from cartera.config import SEMILLA
from cartera.diseno import BLOQUE, CAT, NUM

np.random.seed(SEMILLA)

print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
saldos, movs = datos.leer_todo()
conj = dis.conjuntos(saldos, movs)
subent, val, ent = conj["subentrenamiento"], conj["validación"], conj["entrenamiento completo"]
X, y = ent[NUM + CAT], ent.tardia
print("Entrenamiento completo:", len(ent), "facturas |", round(y.mean() * 100, 1), "% tardías")

Entrenamiento completo: 8930 facturas | 30.2 % tardías


## 1. Pliegues de la validación cruzada agrupada

Cinco pliegues de ventana creciente (D-26). Cada uno valida un trimestre completo y entrena con los
anteriores, solo con facturas que ya habían tenido 30 días después de vencer cuando empieza el
trimestre validado (D-17 en cada corte). Un trimestre nunca está a la vez en entrenamiento y
validación.

In [3]:
cv = dis.pliegues(ent, n=5)
trimestre = ent.fecha.dt.to_period(BLOQUE)
filas = []
for k, (i_ent, i_val) in enumerate(cv, 1):
    assert not set(trimestre.iloc[i_ent]) & set(trimestre.iloc[i_val])
    inicio = trimestre.iloc[i_val].iloc[0].start_time
    assert (ent.fecha_vencimiento.iloc[i_ent] + pd.Timedelta(days=30) <= inicio).all()
    filas.append({"pliegue": k, "trimestre validado": str(trimestre.iloc[i_val].iloc[0]),
                  "facturas entrenamiento": len(i_ent), "facturas validación": len(i_val),
                  "% tardías validación": round(y.iloc[i_val].mean() * 100, 1)})
pd.DataFrame(filas).set_index("pliegue")

,trimestre validado,facturas entrenamiento,facturas validación,% tardías validación
pliegue,,,,
1,2024Q2,5323,704,31.0
2,2024Q3,5972,749,30.2
3,2024Q4,6720,744,31.5
4,2025Q1,7477,698,30.2
5,2025Q2,8199,470,31.9


## 2. Candidatos con su configuración por defecto

Todos con las mismas 11 predictoras (D-24 y D-27, sin `empresa`) y la misma CV. Los que necesitan escala o no aceptan
faltantes (logística, SVM) imputan, escalan y codifican dentro del `Pipeline`; los árboles y
ensambles solo codifican. El desbalance se trata con `class_weight = "balanced"`, que solo afecta al
entrenamiento de cada pliegue.

- **Línea base:** `DummyClassifier` (proporción de tardías).
- **Árbol de referencia de R12** (D-25): profundidad 6, mínimo 50 facturas por hoja.
- **Regresión logística**, **SVM RBF**, **bosque aleatorio** y **gradient boosting**, con los
  valores por defecto de scikit-learn.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

def armar(nombre, num=None):
    """Pipeline de cada candidato con su configuración por defecto (num: para agregar ruido en §5)."""
    lineal, arbol = dis.prep_lineal(num=num), dis.prep_arbol(num=num)
    estimadores = {
        "Línea base (DummyClassifier)": (lineal, DummyClassifier(strategy="prior")),
        "Árbol de referencia (R12)": (arbol, DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                                                    class_weight="balanced", random_state=SEMILLA)),
        "Regresión logística": (lineal, LogisticRegression(class_weight="balanced", max_iter=2000)),
        "SVM RBF": (lineal, SVC(kernel="rbf", class_weight="balanced")),
        "Bosque aleatorio": (arbol, RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEMILLA)),
        "Gradient boosting": (arbol, HistGradientBoostingClassifier(class_weight="balanced", random_state=SEMILLA)),
    }
    prep, est = estimadores[nombre]
    return Pipeline([("prep", prep), ("est", est)])

CANDIDATOS = ["Línea base (DummyClassifier)", "Árbol de referencia (R12)", "Regresión logística",
              "SVM RBF", "Bosque aleatorio", "Gradient boosting"]

def evaluar(modelo):
    r = cross_validate(modelo, X, y, cv=cv, scoring=["average_precision", "roc_auc"])
    return r

resultados = {nombre: evaluar(armar(nombre)) for nombre in CANDIDATOS}

def resumen(r):
    return {"AUC-PR media": r["test_average_precision"].mean(), "AUC-PR desv.": r["test_average_precision"].std(),
            "AUC-ROC media": r["test_roc_auc"].mean(), "segundos de ajuste (media)": r["fit_time"].mean()}

tabla = pd.DataFrame({n: resumen(r) for n, r in resultados.items()}).T
tabla.round(3)

,AUC-PR media,AUC-PR desv.,AUC-ROC media,segundos de ajuste (media)
Línea base (DummyClassifier),0.309,0.007,0.500,0.007
Árbol de referencia (R12),0.665,0.030,0.801,0.013
Regresión logística,0.720,0.021,0.841,0.011
SVM RBF,0.700,0.037,0.840,0.375
Bosque aleatorio,0.697,0.048,0.833,0.104
Gradient boosting,0.700,0.041,0.835,0.623


In [5]:
pd.DataFrame({n: r["test_average_precision"] for n, r in resultados.items()},
             index=[f"pliegue {k}" for k in range(1, 6)]).round(3)

,Línea base (DummyClassifier),Árbol de referencia (R12),Regresión logística,SVM RBF,Bosque aleatorio,Gradient boosting
pliegue 1,0.310,0.633,0.683,0.650,0.622,0.626
pliegue 2,0.302,0.644,0.719,0.663,0.692,0.686
pliegue 3,0.315,0.650,0.726,0.737,0.698,0.716
pliegue 4,0.302,0.685,0.724,0.714,0.700,0.730
pliegue 5,0.319,0.715,0.749,0.736,0.773,0.742


## 3. Búsqueda de hiperparámetros de los ensambles

Búsqueda aleatoria de 30 combinaciones por ensamble, con la misma CV y todas las filas de
entrenamiento (D-22). La mejor de 30 combinaciones tiende a sobrestimar su desempeño (optimismo del
ganador), por eso se compara contra el modelo por defecto: si la diferencia está dentro de la
desviación, es un empate.

In [6]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

espacios = {
    "Gradient boosting": {
        "est__learning_rate": loguniform(0.02, 0.3),
        "est__max_leaf_nodes": [7, 15, 31, 63],
        "est__min_samples_leaf": [20, 50, 100, 200],
        "est__l2_regularization": loguniform(1e-3, 10),
        "est__max_iter": [100, 200, 400],
    },
    "Bosque aleatorio": {
        "est__n_estimators": [200, 400],
        "est__max_depth": [None, 6, 10, 16],
        "est__min_samples_leaf": [1, 5, 20, 50],
        "est__max_features": ["sqrt", 0.5, None],
    },
}
busquedas = {}
for nombre, espacio in espacios.items():
    modelo = armar(nombre)
    if nombre == "Bosque aleatorio":
        modelo.set_params(est__n_jobs=1)   # el paralelismo lo hace la búsqueda
    inicio = time.perf_counter()
    b = RandomizedSearchCV(modelo, espacio, n_iter=30, cv=cv, scoring="average_precision",
                           random_state=SEMILLA, n_jobs=-1).fit(X, y)
    busquedas[nombre] = (b, time.perf_counter() - inicio)

filas = []
for nombre, (b, segundos) in busquedas.items():
    i = b.best_index_
    filas.append({"ensamble": nombre,
                  "por defecto: AUC-PR media": resultados[nombre]["test_average_precision"].mean(),
                  "por defecto: desv.": resultados[nombre]["test_average_precision"].std(),
                  "ajustado: AUC-PR media": b.cv_results_["mean_test_score"][i],
                  "ajustado: desv.": b.cv_results_["std_test_score"][i],
                  "minutos de búsqueda": segundos / 60})
pd.DataFrame(filas).set_index("ensamble").round(3)

,por defecto: AUC-PR media,por defecto: desv.,ajustado: AUC-PR media,ajustado: desv.,minutos de búsqueda
ensamble,,,,,
Gradient boosting,0.700,0.041,0.720,0.043,0.196
Bosque aleatorio,0.697,0.048,0.719,0.049,0.839


In [7]:
for nombre, (b, _) in busquedas.items():
    print(nombre, "→", {k.replace("est__", ""): (round(v, 4) if isinstance(v, float) else v)
                        for k, v in b.best_params_.items()})

Gradient boosting → {'l2_regularization': np.float64(0.0016), 'learning_rate': np.float64(0.0668), 'max_iter': 100, 'max_leaf_nodes': 15, 'min_samples_leaf': 100}
Bosque aleatorio → {'n_estimators': 400, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 16}


Los ajustados se vuelven a evaluar con `cross_validate` en los mismos pliegues, para tener sus
tiempos y sus valores por pliegue en la misma tabla que los demás.

In [8]:
for nombre, (b, _) in busquedas.items():
    ajustado = armar(nombre).set_params(**b.best_params_)
    resultados[f"{nombre} (ajustado)"] = evaluar(ajustado)
tabla = pd.DataFrame({n: resumen(r) for n, r in resultados.items()}).T
tabla.sort_values("AUC-PR media", ascending=False).round(3)

,AUC-PR media,AUC-PR desv.,AUC-ROC media,segundos de ajuste (media)
Regresión logística,0.720,0.021,0.841,0.011
Gradient boosting (ajustado),0.720,0.043,0.845,0.409
Bosque aleatorio (ajustado),0.719,0.049,0.842,0.373
SVM RBF,0.700,0.037,0.840,0.375
Gradient boosting,0.700,0.041,0.835,0.623
Bosque aleatorio,0.697,0.048,0.833,0.104
Árbol de referencia (R12),0.665,0.030,0.801,0.013
Línea base (DummyClassifier),0.309,0.007,0.500,0.007


## 4. Modelo final preliminar

**Regla (D-26, fijada antes de ver resultados):** gana el de mejor AUC-PR media. Si su mejora sobre
la regresión logística (el candidato más simple que no es la línea base) no supera la desviación de
la CV del ganador, gana la logística.

In [9]:
sin_base = tabla.drop(index="Línea base (DummyClassifier)")
mejor = sin_base["AUC-PR media"].idxmax()
ap = {n: r["test_average_precision"] for n, r in resultados.items()}
mejora = ap[mejor].mean() - ap["Regresión logística"].mean()
desv = ap[mejor].std()
dif_pliegue = ap[mejor] - ap["Regresión logística"]
ELEGIDO = mejor if (mejor != "Regresión logística" and mejora > desv) else "Regresión logística"
pd.Series({
    "Mejor AUC-PR media": mejor,
    "Mejora sobre la logística (media)": round(mejora, 3),
    "Desviación de la CV del mejor": round(desv, 3),
    "¿La mejora supera la desviación?": bool(mejora > desv),
    "Pliegues en que el mejor le gana a la logística": f"{int((dif_pliegue > 0).sum())} de 5",
    "Modelo final preliminar": ELEGIDO,
}, dtype=object).to_frame("valor")

,valor
Mejor AUC-PR media,Regresión logística
Mejora sobre la logística (media),0.0
Desviación de la CV del mejor,0.021
¿La mejora supera la desviación?,False
Pliegues en que el mejor le gana a la logística,0 de 5
Modelo final preliminar,Regresión logística


## 5. Importancia de las variables por permutación

El modelo elegido se entrena con el **subentrenamiento** de R12 (2022-01 a 2024-06) y la importancia
se mide en la **validación** de R12 (2024-07 a 2025-06), que el modelo no vio: cuánto cae la AUC-PR
al desordenar cada columna. Se agrega una columna de **ruido** (números al azar) como control: su
importancia debe ser casi nula. Se compara con la importancia por impureza (Gini) del bosque
aleatorio entrenado igual, que es la medida por impureza disponible entre los ensambles.

In [10]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(SEMILLA)
X_sub = subent[NUM + CAT].assign(ruido=rng.normal(size=len(subent)))
X_val = val[NUM + CAT].assign(ruido=rng.normal(size=len(val)))
y_sub, y_val = subent.tardia, val.tardia

def con_parametros(nombre, num):
    base = nombre.replace(" (ajustado)", "")
    modelo = armar(base, num=num)
    if nombre.endswith("(ajustado)"):
        modelo.set_params(**busquedas[base][0].best_params_)
    return modelo

elegido = con_parametros(ELEGIDO, NUM + ["ruido"]).fit(X_sub, y_sub)
perm = permutation_importance(elegido, X_val, y_val, scoring="average_precision",
                              n_repeats=10, random_state=SEMILLA, n_jobs=-1)
permutacion = pd.Series(perm.importances_mean, index=X_val.columns)

bosque = con_parametros("Bosque aleatorio", NUM + ["ruido"]).fit(X_sub, y_sub)
gini = pd.Series(bosque.named_steps["est"].feature_importances_,
                 index=bosque.named_steps["prep"].get_feature_names_out())
# Las categóricas se codifican en varias columnas: se suman por variable original
origen = {c: next((v for v in CAT if c.startswith(v + "_")), c) for c in gini.index}
gini = gini.groupby(origen).sum()

imp = pd.DataFrame({"permutación (caída de AUC-PR)": permutacion, "impureza (Gini, bosque)": gini})
imp["puesto permutación"] = imp.iloc[:, 0].rank(ascending=False).astype(int)
imp["puesto impureza"] = imp.iloc[:, 1].rank(ascending=False).astype(int)
print("Modelo:", ELEGIDO)
imp.sort_values("puesto permutación").round(4)

Modelo: Regresión logística


,permutación (caída de AUC-PR),"impureza (Gini, bosque)",puesto permutación,puesto impureza
atraso_prom_hist,0.2356,0.2165,1,1
n_vencidas_t,0.1081,0.0735,2,7
n_pagadas_hist,0.0662,0.1085,3,3
plazo_dias_observado,0.0161,0.0401,4,11
antiguedad_dias,0.0085,0.0931,5,5
unidad_negocio,0.0033,0.0528,6,10
saldo_abierto_t,0.0005,0.1129,7,2
ruido,0.0003,0.0863,8,6
n_abiertas_t,0.0001,0.0549,9,9
log_monto_inicial,-0.0000,0.1002,10,4


## 6. Error por grupo con un cupo declarado

**Cupo:** se marca el 30 % de las facturas de validación con mayor puntaje (≈ la proporción de
tardías). **Exhaustividad** de un grupo = de sus facturas tardías, qué parte quedó marcada; su
complemento es la **tasa de falsos negativos** (tardías que el modelo dejó pasar). Grupos (D-26):
clientes con 4 o menos facturas pagadas antes de la emisión frente al resto, y Empresa A frente a las
demás. `empresa` no es predictora (D-27), pero se usa aquí para auditar.

In [11]:
def puntaje(modelo, Xd):
    est = modelo.named_steps["est"]
    return modelo.predict_proba(Xd)[:, 1] if hasattr(est, "predict_proba") else modelo.decision_function(Xd)

p = puntaje(elegido, X_val)
marcada = p >= np.quantile(p, 0.70)
grupos = {
    "Historia: 4 o menos pagadas": val.n_pagadas_hist <= 4,
    "Historia: 5 o más pagadas": val.n_pagadas_hist > 4,
    "Empresa A": val.empresa == "Empresa A",
    "Empresas B, C y D": val.empresa != "Empresa A",
}
filas = []
for g, m in grupos.items():
    tardias = m & (y_val == 1)
    filas.append({"grupo": g, "facturas": int(m.sum()), "tardías": int(tardias.sum()),
                  "% marcadas": round(marcada[m].mean() * 100, 1),
                  "exhaustividad": round(marcada[tardias].mean(), 3),
                  "tasa de falsos negativos": round(1 - marcada[tardias].mean(), 3)})
filas.append({"grupo": "Todas", "facturas": len(val), "tardías": int(y_val.sum()),
              "% marcadas": round(marcada.mean() * 100, 1),
              "exhaustividad": round(marcada[y_val == 1].mean(), 3),
              "tasa de falsos negativos": round(1 - marcada[y_val == 1].mean(), 3)})
pd.DataFrame(filas).set_index("grupo")

,facturas,tardías,% marcadas,exhaustividad,tasa de falsos negativos
grupo,,,,,
Historia: 4 o menos pagadas,281,87,52.3,0.759,0.241
Historia: 5 o más pagadas,2380,734,27.4,0.651,0.349
Empresa A,2374,764,31.4,0.683,0.317
"Empresas B, C y D",287,57,18.5,0.386,0.614
Todas,2661,821,30.0,0.663,0.337
